In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import scipy.sparse as sp
import sys
sys.path.append('..')
from src.feature_engineering import build_tfidf_features, extract_manual_features, combine_features

# Đọc dữ liệu
df = pd.read_csv('../data/raw/combined_data.csv')

# XEM TÊN CỘT
print("Tên các cột:", df.columns.tolist())

# THỬ CÁC CÁCH ĐỌC TEXT
# Cách 1: Nếu cột tên là 'text'
if 'text' in df.columns:
    texts = df['text'].fillna('').tolist()
# Cách 2: Nếu cột tên là 'email' hoặc 'content'
elif 'email' in df.columns:
    texts = df['email'].fillna('').tolist()
elif 'content' in df.columns:
    texts = df['content'].fillna('').tolist()
else:
    # Lấy cột đầu tiên (không phải label)
    text_col = [c for c in df.columns if c != 'label'][0]
    texts = df[text_col].fillna('').tolist()
    print(f"Dùng cột: {text_col}")

labels = df['label'].tolist()

print(f"Số dòng: {len(texts)}")
print(f"Mẫu text đầu tiên: {texts[0][:200]}...")
print(f"Nhãn đầu tiên: {labels[0]}")

# Chạy feature với 5000 dòng cho nhanh
df_small = df.head(5000)
if 'text' in df.columns:
    texts_small = df_small['text'].fillna('').tolist()
else:
    text_col = [c for c in df.columns if c != 'label'][0]
    texts_small = df_small[text_col].fillna('').tolist()
labels_small = df_small['label'].tolist()

print("\n1. Đang tạo TF-IDF features...")
tfidf_matrix, vectorizer = build_tfidf_features(texts_small, max_features=2000)

print("2. Đang tạo manual features...")
manual_df = extract_manual_features(texts_small)

print("3. Đang ghép features...")
combined_features = combine_features(tfidf_matrix, manual_df)

X_train, X_test, y_train, y_test = train_test_split(
    combined_features, labels_small, test_size=0.2, random_state=42
)

print("4. Đang train model...")
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\n Độ chính xác: {accuracy:.4f}")

Tên các cột: ['label', 'text']
Số dòng: 83448
Mẫu text đầu tiên: ounce feather bowl hummingbird opec moment alabaster valkyrie dyad bread flack desperate iambic hadron heft quell yoghurt bunkmate divert afterimage...
Nhãn đầu tiên: 1

1. Đang tạo TF-IDF features...
2. Đang tạo manual features...
3. Đang ghép features...
4. Đang train model...

✅ Độ chính xác: 0.9720
